### Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"

### Unsloth

In [2]:
%env UNSLOTH_RETURN_LOGITS = 1 # Run this to disable CCE since it is not supported for CPT

env: UNSLOTH_RETURN_LOGITS=1 # Run this to disable CCE since it is not supported for CPT


In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 16384 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "ornith-ai/Ornith-1.5-9B", # Choose ANY! eg teknium/OpenHermes-2.5-Mistral-7B
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/imrui/miniforge3/envs/tornith/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.19: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 5080. Num GPUs = 1. Max memory: 15.5 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 760/760 [00:02<00:00, 379.87it/s] 


We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

We also add `embed_tokens` and `lm_head` to allow the model to learn out of distribution data.

In [4]:
from pathlib import Path
import yaml

CONFIG_PATH = Path(
    "ornith_hyperparam.yaml"
)

with CONFIG_PATH.open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

training_config = config["training"]
lora_config = config["lora"]
logging_config = config["logging"]

max_seq_length = training_config["max_seq_length"]

print("Loaded configuration:", CONFIG_PATH)
print("Max sequence length:", max_seq_length)
print("Learning rate:", training_config["learning_rate"])

Loaded configuration: ornith_hyperparam.yaml
Max sequence length: 16384
Learning rate: 5e-05


In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_config["lora_r"],
    target_modules=lora_config["target_modules"],
    lora_alpha=lora_config["lora_alpha"],
    lora_dropout=lora_config["lora_dropout"],
    bias="none",
    use_gradient_checkpointing=training_config[
        "gradient_checkpointing"
    ],
    random_state=training_config["random_seed"],
    use_rslora=lora_config["use_rslora"],
    loftq_config={} if lora_config["use_loftq"] else None,
)

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


<a name="Data"></a>
### Data Prep

We only use 1% of the dataset to speed things up! Use more for longer runs!

In [6]:
EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN

import re

HEADING_PATTERN = re.compile(
    r"^(#{1,6})\s+(.+?)\s*$",
    re.MULTILINE,
)

def extract_first_section(text):
    match = HEADING_PATTERN.search(text)

    if match is None:
        return "unknown"

    return match.group(2).strip()

def formatting_prompts_func(examples):
    outputs = []

    for text, meta in zip(examples["text"], examples["meta"]):
        source = meta.get("source", "unknown")
        window = meta.get("window", "unknown")
        section = extract_first_section(text)

        formatted = (
            f"# Source: {source}\n"
            f"# Section: {section}\n"
            f"# Window: {window}\n\n"
            f"{text}"
        )

        if not formatted.rstrip().endswith(EOS_TOKEN):
            formatted += EOS_TOKEN

        outputs.append(formatted)

    return {"text": outputs}
pass

In [7]:
from datasets import load_dataset

DATA_DIR = (
    "/home/imrui/Documents/Jaseci/JacLLM/JacCoder/"
    "dataset/CPT/Ayush-ground-truth"
)

dataset = load_dataset(
    "json",
    data_files={
        "train": [
            f"{DATA_DIR}/train.jsonl",
            f"{DATA_DIR}/valid.jsonl",
        ],
    },
    split="train",
)

# debug
dataset = dataset.train_test_split(train_size = 0.01)["train"]

print(dataset)

dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
    desc="Formatting CPT data",
)

Dataset({
    features: ['text', 'meta'],
    num_rows: 6
})


<a name="Train"></a>
### Continued Pretraining
Now let's use Unsloth's `UnslothTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We do 20 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

Also set `embedding_learning_rate` to be a learning rate at least 2x or 10x smaller than `learning_rate` to make continual pretraining work!

In [8]:
from transformers import TrainingArguments
from unsloth import (
    UnslothTrainer,
    UnslothTrainingArguments,
    is_bfloat16_supported,
)

# Hugging Face uses max_steps <= 0 to mean “train by epochs.”
configured_max_steps = training_config["max_steps"]
max_steps = configured_max_steps if configured_max_steps > 0 else -1

# Embedding and lm_head adapters generally need a lower learning rate.
embedding_learning_rate = training_config["embedding_learning_rate"]
if embedding_learning_rate is None:
    embedding_learning_rate = training_config["learning_rate"] / 10

# Configure logging integrations from the YAML.
report_to = []

if logging_config["enable_wandb"]:
    report_to.append("wandb")

if logging_config["enable_tensorboard"]:
    report_to.append("tensorboard")

if not report_to:
    report_to = "none"

training_args = UnslothTrainingArguments(
    per_device_train_batch_size=training_config["batch_size"],
    gradient_accumulation_steps=training_config[
        "gradient_accumulation_steps"
    ],

    num_train_epochs=training_config["num_epochs"],
    max_steps=max_steps,
    warmup_steps=training_config["warmup_steps"],

    learning_rate=training_config["learning_rate"],
    embedding_learning_rate=embedding_learning_rate,

    optim=training_config["optim"],
    weight_decay=training_config["weight_decay"],
    lr_scheduler_type=training_config["lr_scheduler_type"],

    logging_steps=logging_config["log_frequency"],
    save_steps=training_config["save_steps"],

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),

    seed=training_config["random_seed"],
    output_dir="outputs",

    # All files are being used for CPT training.
    eval_strategy="no",

    report_to=report_to,
    logging_dir=logging_config["tensorboard_dir"],
)

trainer = UnslothTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=training_config["max_seq_length"],
    dataset_num_proc=4,
    packing=training_config["packing"],
    args=training_args,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
Unsloth: Tokenizing ["text"] (num_proc=4): 100%|██████████| 6/6 [00:00<00:00,  6.10 examples/s]


In [9]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA GeForce RTX 5080. Max memory = 15.5 GB.
8.434 GB of memory reserved.


In [11]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6 | Num Epochs = 3 | Total steps = 3
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 10
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 10 x 1) = 10
 "-____-"     Trainable parameters = 232,783,872 of 9,642,597,616 (2.41% trained)


Step,Training Loss


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-3/tokenizer_config.json.


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("Ornith-1-5-9B-CPT-Adapter")  # Local saving
tokenizer.save_pretrained("Ornith-1-5-9B-CPT-Adapter")
# model.push_to_hub("your_name/mistral_v0_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/mistral_v0_lora", token = "YOUR_HF_TOKEN") # Online saving

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("mistral_v0_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/mistral_v0_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False: model.save_pretrained_merged("mistral_v0_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/mistral_v0_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("mistral_v0_lora")
    tokenizer.save_pretrained("mistral_v0_lora")
if False:
    model.push_to_hub("HF_USERNAME/mistral_v0_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/mistral_v0_lora", token = "YOUR_HF_TOKEN")